# Height And Weight Linear Regression

Goal of this project is to finalize Simple Linear Regression on a synthetic dataset of height and weight with one dependent (Height) and independent (Weight) variable

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# reading the dataset
df = pd.read_csv("../../../data/height_weight_synthetic_dataset.csv")
df.head()

In [ ]:
plt.scatter(df['weight_kg'],df['height_cm'])
plt.xlabel("Weight")
plt.ylabel("Height")
plt.title('Scatter Plot')
plt.show()

In [ ]:
# goal: make the best fit line
# STEPS:
# 1. dataset clean and analyse
# 2. Train Test Split
# 3. Standardize and Train
# 4. Evaluate

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
sns.histplot(df["weight_kg"])

In [ ]:
sns.histplot(df["height_cm"],kde=True)

In [ ]:
sns.pairplot(df)

In [ ]:
cols = df.columns
mask = pd.Series(True, index=df.index)
for col in df.columns.drop('height_cm'):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3-Q1

    lower_fence = Q1-1.5*IQR
    higher_fence = Q3+1.5*IQR

    min_val = df[col].min()
    median = df[col].median()
    max_val = df[col].max()

    print(f"{col} 5-number summary:")
    print(f"Min: {min_val}, Q1: {Q1}, Median: {median}, Q3: {Q3}, Max: {max_val}\n")

    # df = df[(df['col']>=lower_fence) & (df['col']<=higher_fence)]
    # unsafe approach using masks:

    mask &= df[col].between(lower_fence, higher_fence)

df_cleaned = df[mask]

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.info()

In [ ]:
# same shit using inbuilt functions:
# Define target column
target_col = 'height_cm'

# Select only feature columns for cleaning
feature_cols = df.drop(columns=target_col)

# Compute 5-number summary for all features at once
summary = feature_cols.describe().loc[['min', '25%', '50%', '75%', 'max']]
print("5-number summary for features:")
print(summary, "\n")

# Compute IQR for all features
Q1 = feature_cols.quantile(0.25)
Q3 = feature_cols.quantile(0.75)
IQR = Q3 - Q1

# Compute lower and upper fences
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

# Create mask for all feature columns at once
mask = feature_cols.apply(lambda x: x.between(lower_fence[x.name], upper_fence[x.name]))

# Combine mask across columns: keep rows where all features are within IQR range
mask_all = mask.all(axis=1)

# Apply mask to original dataframe (target stays intact)
df_cleaned = df[mask_all]
# Rows that were removed (outliers)
df_dropped = df[~mask_all]  # ~ inverts the boolean mask

print("Dropped rows (outliers):")
print(df_dropped)


In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.info()

In [ ]:
sns.boxplot(df)

In [ ]:
sns.boxplot(df_cleaned)

In [ ]:
# now train test split
from sklearn.model_selection import train_test_split

In [ ]:
x=df_cleaned.drop('height_cm',axis=1)
y=df_cleaned['height_cm']
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.25,random_state=42)

In [ ]:
x_train.head()

In [ ]:
x.shape

In [ ]:
x_train.shape, x_test.shape, y_train.shape, y_test.shape

In [ ]:
# Standardizing the Train independent feature:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
x_train_scaled = scaler.fit_transform(x_train)
x_train_scaled

In [ ]:
x_test_scaled = scaler.transform(x_test)
x_test_scaled

In [ ]:
plt.scatter(x_train_scaled,y_train)

In [ ]:
# Train the model
from sklearn.linear_model import LinearRegression

In [ ]:
linear_regression = LinearRegression()

In [ ]:
linear_regression.fit(x_train_scaled,y_train)

In [ ]:
linear_regression.get_params()

In [ ]:
print(f"Slope: {linear_regression.coef_}")
print(f"Intercept: {linear_regression.intercept_}")

In [ ]:
plt.scatter(x_train_scaled,y_train)
plt.plot(x_train_scaled,linear_regression.predict(x_train_scaled),color='r')

In [ ]:
# prediction of train data:
# intercept + coef_(weights)
#prediction of test data:
# intercept + coef_(weights)

y_pred_test = linear_regression.predict(x_test_scaled)

In [ ]:
y_pred_test,y_test

In [ ]:
plt.scatter(x_test_scaled,y_test)
plt.plot(x_test_scaled,linear_regression.predict(x_test_scaled),color='r')

In [ ]:
# evaluation

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
mse = mean_squared_error(y_test,y_pred_test)
mae = mean_absolute_error(y_test,y_pred_test)
rmse = np.sqrt(mse)
print(f'MSE: {mse} \nMAE: {mae} \nRMSE: {rmse}')

In [ ]:
scr = r2_score(y_test,y_pred_test)
print(f"r2 score: {scr}")

In [ ]:
scaled_weight = scaler.transform([[80]])
scaled_weight

In [ ]:
linear_regression.predict(scaled_weight)

In [ ]:
# Assumption:
plt.scatter(y_test,y_pred_test)

In [ ]:
# almost linear means good model

In [ ]:
residuals = y_test-y_pred_test
residuals

In [ ]:
sns.histplot(residuals,kde=True)

In [ ]:
# scatter plot with predictions and residual should be uniform
plt.scatter(y_pred_test,residuals)

In [ ]:
# pickling the model:
